Criação de um modelo baseline para comparação de performance

IMPORTANTE: DEVE SER RECALCULADO UMA VEZ QUE OS DADOS DE TESTE FOREM DISPONIBILIZADOS

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline

# 1. Upload dos dados

In [ ]:
# Usando base de treino original e sem duplicatas
df = pd.read_csv("../data/train.csv")
# df = pd.read_csv("../data/train_noDuplicates.csv")
# df = pd.read_csv("../data/train_noSameClassDuplicates.csv")
df

,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234
...,...,...
20087,Prezado Sr. Luciano Wolff A sua solicitação ...,c1
20088,"PREZADO CIDADãO, A COORDENAçãO DO PROGRAMA FA...",c1
20089,"Prezado, Informamos que o Ministério da Agr...",c1
20090,"Prezado Senhor, Encaminhamos em anexo respos...",c1


# 2. Tokenização e contagens de frequência

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer()
X = vect.fit_transform(df["resp_text"]) # a coluna de texto a vetorizar
print(X.shape)

(20092, 52339)


In [4]:
Y = df["clarity"]
print(Y.shape)

(20092,)


# 3. Separação treinamento/teste

In [5]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size=0.2,
random_state=123, stratify=Y)

# 4. Dummy Classifier

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report, accuracy_score

clf = DummyClassifier(strategy='most_frequent', random_state = 100,
constant = None).fit(x_train, y_train)

In [7]:
predicted = clf.predict(x_test)
score = f1_score(y_test,predicted,average='macro')
accuracy = accuracy_score(y_test, predicted)

print("\n\nMaj =====> {:2.2f}\n".format(score).replace(".",","))
print("Accuracy =====> {:2.2f}".format(accuracy).replace(".",","))



Maj =====> 0,17

Accuracy =====> 0,34


In [8]:
from sklearn.pipeline import Pipeline

pipelineDummy = Pipeline(
    steps=[
        ("clf", DummyClassifier(strategy='most_frequent', random_state = 100,
constant = None))
    ]
)

In [9]:
pipelineDummy.fit(x_train, y_train)

,steps,"[('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,strategy,'most_frequent'
,random_state,100
,constant,None


In [10]:
y_pred = pipelineDummy.predict(x_test)

In [11]:
score = f1_score(y_test, y_pred, average='macro')
accuracy = accuracy_score(y_test, y_pred)

print("\n\nMaj =====> {:2.2f}\n".format(score).replace(".", ","))
print("Accuracy =====> {:2.2f}".format(accuracy).replace(".", ","))



Maj =====> 0,17

Accuracy =====> 0,34


In [12]:
import joblib

joblib.dump(pipelineDummy, "pipelines/pipelineDummy.pkl")

['pipelines/pipelineDummy.pkl']

# 5. Regressão Logística sobre contagens TF-IDF

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [14]:
vect = TfidfVectorizer ()
X = vect.fit_transform(df["resp_text"])

In [15]:
x_train, x_test, y_train, y_test = train_test_split(X, Y,test_size=0.2,random_state=123,stratify=Y)

In [16]:
clf = LogisticRegression(class_weight='balanced').fit(x_train, y_train)

In [17]:
predicted = clf.predict(x_test)
score = f1_score(y_test,predicted,average='macro')
accuracy = accuracy_score(y_test, predicted)

print("\n\nTF-IDF =====> {:2.2f}\n".format(score).replace(".",","))
print("Accuracy =====> {:2.2f}".format(accuracy).replace(".",","))



TF-IDF =====> 0,46

Accuracy =====> 0,47


In [18]:
from sklearn.pipeline import Pipeline

pipelineRegLogTFIDF = Pipeline(
    steps=[
        ("clf", LogisticRegression(class_weight='balanced'))
    ]
)

pipelineRegLogTFIDF.fit(x_train, y_train)

,steps,"[('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'


In [19]:
y_pred = pipelineRegLogTFIDF.predict(x_test)
score = f1_score(y_test, y_pred, average='macro')
accuracy = accuracy_score(y_test, y_pred)

print("\n\nTF-IDF =====> {:2.2f}\n".format(score).replace(".",","))
print("Accuracy =====> {:2.2f}".format(accuracy).replace(".",","))



TF-IDF =====> 0,46

Accuracy =====> 0,47


In [20]:
import joblib

joblib.dump(pipelineRegLogTFIDF, "pipelines/pipelineRegLogTFIDF.pkl")

['pipelines/pipelineRegLogTFIDF.pkl']